In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import (classification_report, confusion_matrix, 
                             roc_auc_score, average_precision_score,
                             precision_recall_curve, RocCurveDisplay)
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import shap
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)

# Load feature matrix
df = pd.read_csv('feature_matrix_v2.csv')

print("=== FEATURE MATRIX LOADED ===")
print(f"Shape: {df.shape}")
print(f"Fraud accounts: {df['is_fraud'].sum():,}")
print(f"Legitimate accounts: {(df['is_fraud']==0).sum():,}")
print(f"Fraud rate: {df['is_fraud'].mean()*100:.4f}%")

=== FEATURE MATRIX LOADED ===
Shape: (2768630, 22)
Fraud accounts: 8,213
Legitimate accounts: 2,760,417
Fraud rate: 0.2966%


In [2]:
# Define feature columns
feature_cols = [c for c in df.columns if c not in ['nameOrig', 'is_fraud']]

print(f"Features being used: {len(feature_cols)}")
print(feature_cols)

X = df[feature_cols].values
y = df['is_fraud'].values

# Train/test split — 80/20, stratified to preserve fraud ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"\n=== DATA SPLIT ===")
print(f"Training set:   {X_train.shape[0]:,} accounts")
print(f"Test set:       {X_test.shape[0]:,} accounts")
print(f"Fraud in train: {y_train.sum():,} ({y_train.mean()*100:.4f}%)")
print(f"Fraud in test:  {y_test.sum():,} ({y_test.mean()*100:.4f}%)")

Features being used: 20
['velocity_ratio', 'balance_retention_rate', 'fan_out_degree', 'dormancy_spike_score', 'round_amount_flag', 'round_amount_ratio', 'type_entropy', 'account_age_at_activation', 'fan_in_degree', 'regularity_index', 'max_amount_ratio', 'total_transactions', 'avg_amount', 'max_amount', 'structuring_score', 'freq_acceleration', 'amount_variance_score', 'session_burst_score', 'dormancy_ratio', 'onboarding_velocity']

=== DATA SPLIT ===
Training set:   2,214,904 accounts
Test set:       553,726 accounts
Fraud in train: 6,570 (0.2966%)
Fraud in test:  1,643 (0.2967%)


In [3]:
# RULE ENGINE BASELINE
# Apply R1-R5 rules to test set and measure performance

test_df = df.iloc[X_test.shape[0] * -1:].copy() if False else df.sample(frac=0.2, random_state=42)

# Rebuild test set with account names for rule application
test_df = pd.DataFrame(X_test, columns=feature_cols)
test_df['is_fraud'] = y_test

def apply_rules(row):
    score = 0
    # R1: High velocity ratio + high amount
    if row['velocity_ratio'] > 0.85:
        score += 1
    # R2: Low fan-out + multiple transactions
    if row['fan_out_degree'] <= 2 and row['total_transactions'] > 1:
        score += 1
    # R3: Dormancy spike
    if row['dormancy_spike_score'] > 0.50:
        score += 1
    # R4: Round amount + low retention
    if row['round_amount_flag'] == 1 and row['balance_retention_rate'] < 0.05:
        score += 1
    # R5: High amount + high velocity
    if row['avg_amount'] > 500000 and row['velocity_ratio'] > 0.70:
        score += 1
    return 1 if score >= 2 else 0

print("Applying rule engine to test set...")
test_df['rule_prediction'] = test_df.apply(apply_rules, axis=1)

# Evaluate
from sklearn.metrics import precision_score, recall_score, f1_score

rule_precision = precision_score(test_df['is_fraud'], test_df['rule_prediction'])
rule_recall    = recall_score(test_df['is_fraud'], test_df['rule_prediction'])
rule_f1        = f1_score(test_df['is_fraud'], test_df['rule_prediction'])

print("\n=== RULE ENGINE BASELINE ===")
print(f"Precision: {rule_precision:.4f}")
print(f"Recall:    {rule_recall:.4f}")
print(f"F1-Score:  {rule_f1:.4f}")
print(f"\nFlagged as mule:    {test_df['rule_prediction'].sum():,}")
print(f"Actual mule:        {test_df['is_fraud'].sum():,}")
print(f"True positives:     {((test_df['rule_prediction']==1) & (test_df['is_fraud']==1)).sum():,}")
print(f"False positives:    {((test_df['rule_prediction']==1) & (test_df['is_fraud']==0)).sum():,}")
print("\nThis is the baseline every ML model must beat.")

Applying rule engine to test set...

=== RULE ENGINE BASELINE ===
Precision: 0.0119
Recall:    0.4547
F1-Score:  0.0231

Flagged as mule:    62,969
Actual mule:        1,643
True positives:     747
False positives:    62,222

This is the baseline every ML model must beat.


## Finding: Rule Engine Baseline

**Result:** Precision 0.0119 | Recall 0.4547 | F1 0.0231

**What this means:**
- The rule engine flagged **62,969 accounts** to catch **747 actual mules**
- For every real mule caught, **83 innocent accounts** were falsely flagged
- This mirrors PricewaterhouseCoopers' finding that legacy AML systems generate **90-95% false positives**
- Proven on our own data — not just a cited statistic

**Why recall is 45% but precision is only 1.19%:**
- The rules are broad by design — they catch many mules but also catch many legitimate accounts with similar transaction patterns
- A legitimate customer making a single large TRANSFER also triggers R1 and R5

**The number to beat:**
- F1-Score: **0.0231**
- This is the floor. Any ML model worth deploying must significantly exceed this.

**Regulatory note:**
- Despite poor precision, the rule engine remains valuable for PMLA compliance
- Every flag has a documented, auditable threshold tied to a regulatory obligation
- SHAP in Phase 2 adds the explainability layer that makes ML flags equally auditable

In [4]:
# SMOTE — fix class imbalance before training
print("Applying SMOTE to training set...")
print(f"Before SMOTE — Fraud: {y_train.sum():,}, Legitimate: {(y_train==0).sum():,}")

smote = SMOTE(random_state=42, sampling_strategy=0.1)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"After SMOTE  — Fraud: {y_train_sm.sum():,}, Legitimate: {(y_train_sm==0).sum():,}")
print(f"New fraud rate: {y_train_sm.mean()*100:.2f}%")
print(f"Training set size: {len(X_train_sm):,}")

Applying SMOTE to training set...
Before SMOTE — Fraud: 6,570, Legitimate: 2,208,334
After SMOTE  — Fraud: 220,833, Legitimate: 2,208,334
New fraud rate: 9.09%
Training set size: 2,429,167


## Finding: SMOTE Class Imbalance Correction

**Before SMOTE:** 6,570 fraud vs 2,208,334 legitimate (0.30% fraud rate)
**After SMOTE:**  220,833 fraud vs 2,208,334 legitimate (9.09% fraud rate)

**What SMOTE does:**
- Generates SYNTHETIC fraud samples by interpolating between existing fraud cases in feature space
- Does NOT duplicate — creates new plausible fraud accounts based on patterns in existing ones
- Applied ONLY to training data — test set remains imbalanced to reflect real-world conditions

**Why 9.09% and not 50%:**
- sampling_strategy=0.1 means fraud becomes 10% of legitimate count
- Going to 50% would over-correct and make the model too aggressive
- 9.09% gives the model enough fraud signal without distorting the decision boundary

**Critical rule:**
- SMOTE on test data = methodological error
- Test set must stay imbalanced to measure real-world performance honestly

In [5]:
# Scale features for Logistic Regression
# XGBoost does not need scaling but LR does

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sm)
X_test_scaled  = scaler.transform(X_test)

print("Features scaled using StandardScaler")
print(f"Training set: {X_train_scaled.shape}")
print(f"Test set:     {X_test_scaled.shape}")

Features scaled using StandardScaler
Training set: (2429167, 20)
Test set:     (553726, 20)


In [6]:
print("Training Logistic Regression...")

lr = LogisticRegression(
    C=1.0,
    class_weight='balanced',
    max_iter=1000,
    random_state=42,
    n_jobs=-1
)

lr.fit(X_train_scaled, y_train_sm)
lr_pred = lr.predict(X_test_scaled)
lr_prob = lr.predict_proba(X_test_scaled)[:, 1]

print("\n=== LOGISTIC REGRESSION RESULTS ===")
print(classification_report(y_test, lr_pred, 
      target_names=['Legitimate', 'Mule Account']))

lr_auc = roc_auc_score(y_test, lr_prob)
lr_ap  = average_precision_score(y_test, lr_prob)
lr_f1  = f1_score(y_test, lr_pred)

print(f"AUC-ROC:           {lr_auc:.4f}")
print(f"Average Precision: {lr_ap:.4f}")
print(f"F1-Score:          {lr_f1:.4f}")
print(f"\nBaseline F1 to beat: 0.0231 (rule engine)")
print(f"Improvement:         {lr_f1/0.0231:.1f}x")

Training Logistic Regression...

=== LOGISTIC REGRESSION RESULTS ===
              precision    recall  f1-score   support

  Legitimate       1.00      0.81      0.90    552083
Mule Account       0.01      0.68      0.02      1643

    accuracy                           0.81    553726
   macro avg       0.50      0.75      0.46    553726
weighted avg       1.00      0.81      0.89    553726

AUC-ROC:           0.7897
Average Precision: 0.0431
F1-Score:          0.0210

Baseline F1 to beat: 0.0231 (rule engine)
Improvement:         0.9x
